# Object Detection Pipeline - Hyperparameter Tuning & Model Training
This notebook demonstrates the complete pipeline for training different YOLO models on aerial imagery. Besides, Hyperparameter tuning with Optuma, followed by final training and evaluation on test set.

Github Repo and Documentation of the work : [DL4CV Coconut Detection](https://github.com/kshitijrajsharma/dl4cv-oda)

By/ Kshitij Raj Sharma, Sahar Mohamed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshitijrajsharma/dl4cv-oda/blob/master/notebooks/pipeline.ipynb)

This dl4cv_oda package includes all the pipline steps and functions for coconut trees, more info in the repo here: [DL4CV Coconut Detection](https://github.com/kshitijrajsharma/dl4cv-oda)

In [ ]:
# ! pip install dl4cv_oda

# Object Detection Models Summary

## Comparison Table

| Model | Type | Key Architecture | Main Innovation | Strengths | Use Case |
|-------|------|-----------------|----------------|-----------|----------|
| **YOLOv8** | CNN-based | Backbone + Neck (FPN/PAN) + Split Head | Anchor-free, C2f modules | High speed, multi-task support | Real-time detection, balanced speed/accuracy |
| **YOLOv12** | CNN + Attention | R-ELAN Backbone + Area Attention | Attention mechanisms in YOLO | Better context, small object detection | Real-time with enhanced accuracy |
| **RT-DETR** | Transformer | Hybrid Encoder + Query Selection | End-to-end, NMS-free | Crowded scenes, global context | Complex scenes, research applications |

## YOLOv8

**Architecture:** Backbone → Neck → Head

**Key Features:**
- Anchor-free detection (direct center prediction)
- C2f modules (replaces C3 blocks)
- Decoupled classification/regression heads
- Multi-task:  detection, segmentation, classification, pose

**Best for:** General-purpose real-time detection

## YOLOv12

**Architecture:** R-ELAN Backbone + Area Attention Module

**Key Features:**
- Area Attention for high-res feature maps
- Residual Efficient Layer Aggregation Networks (R-ELAN)
- Attention-friendly architecture
- Optimized gradient flow

**Best for:** Small/detailed objects with real-time constraints

## RT-DETR

**Architecture:** Hybrid Encoder + Uncertainty-Minimal Query Selection

**Key Features:**
- First real-time Transformer detector
- End-to-end (no NMS, no anchors)
- Hybrid multi-scale encoder
- Fixed set object prediction

**Best for:** Dense/crowded scenes, GPU deployment

**Source:** Ultralytics

In [ ]:
import requests
import geopandas as gpd
import json
import yaml
import time
import torch
import optuna
import pandas as pd
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO, RTDETR
from dl4cv_oda import (clean_osm_data, clip_labels_to_tiles, convert_to_yolo_format,
                       create_train_val_split, create_yolo_config, download_tiles)

## Step 1: Data Preprocessing

In [ ]:
DATA_DIR = Path.cwd().parent / "data"
RAW_DIR = DATA_DIR / "raw"
CHIPS_DIR = DATA_DIR / "chips"
LABELS_DIR = DATA_DIR / "labels"
YOLO_DIR = DATA_DIR / "yolo"

TARGET = 'Coconut'

OSM_FILE = RAW_DIR / "kolovai-trees.geojson"
CLEANED_FILE = RAW_DIR / "cleaned.geojson"
TREES_BOX_FILE = DATA_DIR / "trees_box.geojson"
TILES_FILE = DATA_DIR / "tiles.geojson"

if not OSM_FILE.exists():
    OSM_FILE.parent.mkdir(parents=True, exist_ok=True)
    OSM_FILE.write_bytes(requests.get("https://github.com/kshitijrajsharma/dl4cv-oda/blob/master/data/raw/kolovai-trees.geojson? raw=true", allow_redirects=True).content)
    print(f"Downloaded OSM data")

if not CLEANED_FILE.exists():
    count = clean_osm_data(str(OSM_FILE), str(CLEANED_FILE), str(TREES_BOX_FILE),target=TARGET)
    print(f"Cleaned {count} trees")

if not TILES_FILE.exists():
    data = gpd.read_file(TREES_BOX_FILE)
    data. to_crs(epsg=4326, inplace=True)
    bbox = list(data.total_bounds)
    await download_tiles(bbox, 19, "https://tiles.openaerialmap.org/5a28639331eff4000c380690/0/5b1b6fb2-5024-4681-a175-9b667174f48c/{z}/{x}/{y}.png", DATA_DIR, 'OAM')
    print("Downloaded tiles")

label_stats = {}
if not (YOLO_DIR / "train").exists():
    label_stats = clip_labels_to_tiles(str(TREES_BOX_FILE), str(TILES_FILE), str(LABELS_DIR))
    print(f"Clipped labels to tiles: {label_stats}")
    
    class_mapping = convert_to_yolo_format(str(TREES_BOX_FILE), str(CHIPS_DIR), str(LABELS_DIR), str(YOLO_DIR))
    print(f"Converted to YOLO format")
    
    train_count, val_count, test_count = create_train_val_split(str(LABELS_DIR), str(CHIPS_DIR), str(YOLO_DIR), train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42)
    print(f"Split:  train={train_count}, val={val_count}, test={test_count}")
    
    config_file = create_yolo_config(str(YOLO_DIR), {"Coconut": 0})
    print(f"Config:  {config_file}")

print("Data preparation complete")

## Step 2: Configuration

In [ ]:

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 64
IMG_SIZE = 256
EPOCHS = 200
PATIENCE = 30
BATCH = 16

TUNE_DEFAULT = False
TUNE_OPTUNA = True
TUNE_ITERATIONS = 8
TUNE_EPOCHS = 80
TUNE_PATIENCE = 10

MODELS = [
    {"name": "yolov8l", "weights": "yolov8l.pt"},
    {"name": "yolo12l", "weights": "yolo12l.pt"},
    {"name": "rtdetr-l", "weights": "rtdetr-l.pt"},
]

EXPERIMENT_NAME = "full_pipeline"


torch.manual_seed(SEED)
exp_id = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_NAME = f"{EXPERIMENT_NAME}_{exp_id}" if EXPERIMENT_NAME else exp_id
print(f"Experiment:  {EXPERIMENT_NAME}")
print(f"Models: {[m['name'] for m in MODELS]}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024.0 **3):.2f} GB" if torch.cuda.is_available() else "N/A")


## Helper Functions

In [ ]:
def calculate_metrics(metrics):
    p, r = float(metrics. box. mp), float(metrics.box.mr)
    f1 = 2 * (p * r) / (p + r + 1e-6)
    return {
        'precision': p,
        'recall': r,
        'f1': f1,
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map)
    }

def train_and_evaluate(model, name, run_name, hyperparams=None):
    start_time = time.time()
    
    train_params = {
        'data':  str(YOLO_DIR / "config.yaml"),
        'epochs': EPOCHS,
        'imgsz': IMG_SIZE,
        'patience': PATIENCE,
        'batch': BATCH,
        'seed': SEED,
        'name': run_name,
        'project': 'runs',
        'plots': True,
        'verbose': False,
    }
    
    if hyperparams:
        train_params.update(hyperparams)
    
    model.train(**train_params)
    train_time = time.time() - start_time
    
    val_start = time.time()
    val_metrics = model.val(split='val', verbose=False)
    val_time = time.time() - val_start
    
    test_start = time.time()
    test_metrics = model.val(split='test', verbose=False)
    test_time = time.time() - test_start
    
    return {
        'val':  calculate_metrics(val_metrics),
        'test': calculate_metrics(test_metrics),
        'train_time': train_time,
        'val_inference_time': val_time,
        'test_inference_time': test_time
    }

def tune_with_optuna(model_cfg, name):
    def objective(trial):
        lr0 = trial.suggest_float("lr0", 1e-5, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch = trial.suggest_categorical("batch", [8, 16, 32])
        
        try:
            model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
            model.train(
                data=str(YOLO_DIR / "config.yaml"),
                epochs=TUNE_EPOCHS,
                imgsz=IMG_SIZE,
                batch=batch,
                lr0=lr0,
                weight_decay=weight_decay,
                seed=SEED,
                verbose=False,
                plots=False,
                save=False,
                patience=PATIENCE,
            )
            
            val_metrics = model.val(split='val', verbose=False)
            metrics = calculate_metrics(val_metrics)
            return metrics['f1']
        except Exception as e:
            print(f"Trial failed: {e}")
            return 0.0
    
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=TUNE_ITERATIONS, show_progress_bar=False)
    return study.best_params

## Step 3: Train Base Model

In [ ]:
results = []

for model_cfg in MODELS:
    name = model_cfg['name']
    print(f"\nTraining {name} (base)")
    
    model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
    metrics = train_and_evaluate(model, name, f"{exp_id}_{name}_base")
    
    results.append({
        'model': name,
        'type': 'base',
        'val_precision': metrics['val']['precision'],
        'val_recall': metrics['val']['recall'],
        'val_f1': metrics['val']['f1'],
        'val_map50': metrics['val']['map50'],
        'test_precision': metrics['test']['precision'],
        'test_recall': metrics['test']['recall'],
        'test_f1':  metrics['test']['f1'],
        'test_map50': metrics['test']['map50'],
        'train_time': metrics['train_time'],
        'val_inference_time': metrics['val_inference_time'],
        'test_inference_time': metrics['test_inference_time']
    })
    
    print(f"{name}:  val_f1={metrics['val']['f1']:.4f}, test_f1={metrics['test']['f1']:.4f}, test_map50={metrics['test']['map50']:.4f}")

## Step 4: Hyperparameter Tuning (Default)

In [ ]:
if TUNE_DEFAULT:
    for model_cfg in MODELS:
        name = model_cfg['name']
        print(f"\nTuning {name} (ultralytics)")
        
        model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
        
        model.tune(
            data=str(YOLO_DIR / "config.yaml"),
            epochs=TUNE_EPOCHS,
            iterations=TUNE_ITERATIONS,
            imgsz=IMG_SIZE,
            plots=False,
            save=False,
            val=True
        )
        
        best_cfg_path = Path(f"runs/detect/{name}/best_hyperparameters.yaml")
        tuned_params = {}
        if best_cfg_path.exists():
            with open(best_cfg_path, 'r') as f:
                tuned_params = yaml.safe_load(f)
            if 'close_mosaic' in tuned_params:
                tuned_params['close_mosaic'] = int(tuned_params['close_mosaic'])
        
        model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
        metrics = train_and_evaluate(model, name, f"{exp_id}_{name}_default_tuned", tuned_params)
        
        results.append({
            'model': name,
            'type': 'default_tuned',
            'val_precision': metrics['val']['precision'],
            'val_recall': metrics['val']['recall'],
            'val_f1': metrics['val']['f1'],
            'val_map50':  metrics['val']['map50'],
            'test_precision': metrics['test']['precision'],
            'test_recall': metrics['test']['recall'],
            'test_f1':  metrics['test']['f1'],
            'test_map50': metrics['test']['map50'],
            'train_time': metrics['train_time'],
            'val_inference_time': metrics['val_inference_time'],
            'test_inference_time': metrics['test_inference_time']
        })
        
        exp_results_dir = RESULTS_DIR / exp_id
        exp_results_dir.mkdir(exist_ok=True)
        if best_cfg_path.exists():
            import shutil
            shutil.copy(best_cfg_path, exp_results_dir / f"{name}_default_best_hyperparameters.yaml")
        
        print(f"{name}: val_f1={metrics['val']['f1']:. 4f}, test_f1={metrics['test']['f1']:. 4f}, test_map50={metrics['test']['map50']:. 4f}")

## Step 5: Hyperparameter Tuning (Optuna)

In [ ]:
if TUNE_OPTUNA:
    for model_cfg in MODELS:
        name = model_cfg['name']
        print(f"\nTuning {name} (optuna)")
        
        best_params = tune_with_optuna(model_cfg, name)
        print(f"Best params: {best_params}")
        
        exp_results_dir = RESULTS_DIR / exp_id
        exp_results_dir. mkdir(exist_ok=True)
        with open(exp_results_dir / f"{name}_optuna_best_hyperparameters.yaml", 'w') as f:
            yaml.safe_dump(best_params, f)
        
        model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
        metrics = train_and_evaluate(model, name, f"{exp_id}_{name}_optuna_tuned", best_params)
        
        results.append({
            'model': name,
            'type': 'optuna_tuned',
            'val_precision': metrics['val']['precision'],
            'val_recall': metrics['val']['recall'],
            'val_f1':  metrics['val']['f1'],
            'val_map50': metrics['val']['map50'],
            'test_precision': metrics['test']['precision'],
            'test_recall': metrics['test']['recall'],
            'test_f1': metrics['test']['f1'],
            'test_map50':  metrics['test']['map50'],
            'train_time': metrics['train_time'],
            'val_inference_time': metrics['val_inference_time'],
            'test_inference_time': metrics['test_inference_time']
        })
        
        print(f"{name}: val_f1={metrics['val']['f1']:.4f}, test_f1={metrics['test']['f1']:.4f}, test_map50={metrics['test']['map50']:.4f}")

## Save Results

In [ ]:
import shutil

df = pd.DataFrame(results)

device_memory = torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3) if torch.cuda.is_available() else None

summary = {
    'exp_id': exp_id,
    'seed': SEED,
    'epochs': EPOCHS,
    'img_size': IMG_SIZE,
    'data' : label_stats,
    'batch': BATCH,
    'patience': PATIENCE,
    'tune_default': TUNE_DEFAULT,
    'tune_optuna': TUNE_OPTUNA,
    'tune_iterations': TUNE_ITERATIONS,
    'tune_epochs': TUNE_EPOCHS,
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'device_memory_gb': device_memory,
    'gpu_available': torch.cuda.is_available(),
    'models': [m['name'] for m in MODELS],
    'results': results,
    'best_model': results[df['test_map50'].idxmax()]['model'] if len(results) > 0 else None,
    'best_test_map50': float(df['test_map50'].max()) if len(results) > 0 else 0.0
}

exp_results_dir = RESULTS_DIR / exp_id
exp_results_dir.mkdir(exist_ok=True)

model_metrics = {}
RUN_DIR = Path("runs")

for model_cfg in MODELS:
    name = model_cfg['name']
    model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
    
    total_params = sum(p.numel() for p in model.model.parameters())
    model_size = None
    
    for run_type in ['base', 'default_tuned', 'optuna_tuned']:
        run_name = f"{exp_id}_{name}_{run_type}"
        run_path = RUN_DIR / run_name
        
        if not run_path.exists():
            continue
        
        if model_size is None:
            best_pt = run_path / "weights" / "best.pt"
            if best_pt.exists():
                model_size = best_pt.stat().st_size / (1024.0 ** 2)
        
        model_results_dir = exp_results_dir / name / run_type
        model_results_dir.mkdir(parents=True, exist_ok=True)
        
        for file in ["results.csv","results.png","val_batch1_labels.jpg","val_batch1_pred.jpg","args.yaml"]:
            src = run_path / file
            if src.exists():
                shutil.copy(src, model_results_dir / file)
    
    model_metrics[name] = {
        'total_parameters': total_params,
        'model_size_mb': model_size
    }

summary['model_metrics'] = model_metrics

with open(exp_results_dir / "summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print("\nResults")
print(df[['model', 'type', 'val_f1', 'val_map50', 'test_f1', 'test_map50']].to_string(index=False))
print(f"\nBest: {summary['best_model']} (test_map50={summary['best_test_map50']:.4f})")
print(f"Saved: results/{exp_id}/summary.json")

## Step 6: Visualization

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def show_plot(path, title=None):
    if not path.exists():
        return
    img = Image.open(path)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

RUN_DIR = Path("runs")

for model_cfg in MODELS:
    name = model_cfg['name']
    
    for run_type in ['base', 'default_tuned', 'optuna_tuned']:
        run_name = f"{exp_id}_{name}_{run_type}"
        
        results_path = RUN_DIR / run_name / "results.png"
        if results_path.exists():
            show_plot(results_path, title=f"{name} ({run_type}) - Training Curves")
        
        cm_path = RUN_DIR / run_name / "confusion_matrix.png"
        if cm_path.exists():
            show_plot(cm_path, title=f"{name} ({run_type}) - Confusion Matrix")